# Options from Zero · Episode 11: Random walks and Brownian motion

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **No data.** Every path in this notebook is simulated, with fixed random seeds so the numbers repeat.
>
> Educational material only, not financial advice. Options are risky: a seller can lose far more than the premium received.

1. Setup · 2. From coin flips to paths · 3. The spread grows with the square root of time · 4. Infinitely wiggly · 5. Squared moves add up to time · 6. History · 7. This episode's question · 8. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data files this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import QuantLib as ql

out = {"meta": {
    "series": "Options from Zero",
    "episode": 11,
    "quantlib_version": ql.__version__,
    "quotes_label": "illustrative (made up for teaching)",
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}
S, K, T, r, vol = 100.0, 105.0, 1.0, 0.036, 0.20   # the running trade

The option formulas used throughout the series (`episodes/_shared/options.py`):

In [ ]:
# Option formulas for Options from Zero, written out by hand so every step can be read. Copied verbatim into
# each notebook by make_notebook.py, so every notebook runs on its own; the notebooks check these against
# QuantLib. Needs: math, numpy as np.
#   Running trade (Part 1): an illustrative share at $100, no dividends, 20% volatility, a one-year call struck
#   at $105, interest 3.60% a year continuously compounded. Made up for teaching.

def norm_cdf(x):
    """N(x): the chance a standard normal variable is below x."""
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))

def call_payoff(s, k):
    return np.maximum(np.asarray(s, dtype=float) - k, 0.0)

def put_payoff(s, k):
    return np.maximum(k - np.asarray(s, dtype=float), 0.0)

def black_scholes(s, k, t, r, vol, call=True):
    """Black-Scholes price of a European option on a share with no dividends (r continuously compounded)."""
    d1 = (math.log(s / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    d2 = d1 - vol * math.sqrt(t)
    if call:
        return s * norm_cdf(d1) - k * math.exp(-r * t) * norm_cdf(d2)
    return k * math.exp(-r * t) * norm_cdf(-d2) - s * norm_cdf(-d1)

def one_step(s, up, down, k, t, r, call=True):
    """One-step tree: replicate the option with delta shares and cash. Returns the pieces and the price."""
    pay = call_payoff if call else put_payoff
    v_up, v_down = float(pay(up, k)), float(pay(down, k))
    delta = (v_up - v_down) / (up - down)
    cash = (v_down - delta * down) * math.exp(-r * t)   # negative: borrowed today
    q = (s * math.exp(r * t) - down) / (up - down)       # risk-neutral probability of the up move
    return {"v_up": v_up, "v_down": v_down, "delta": delta, "cash": cash, "price": delta * s + cash, "q": q}

def binomial_tree(s, k, t, r, vol, steps, call=True, american=False, quantlib_p=False):
    """Cox-Ross-Rubinstein tree: u = exp(vol sqrt(dt)), d = 1/u, worked backward from expiry.
    quantlib_p=True uses QuantLib's up-probability 1/2 + 1/2 (r - vol^2/2) sqrt(dt) / vol instead of the textbook
    (e^{r dt} - d) / (u - d); both give the same price in the limit."""
    dt = t / steps
    u = math.exp(vol * math.sqrt(dt)); d = 1 / u
    p = 0.5 + 0.5 * (r - vol ** 2 / 2) * math.sqrt(dt) / vol if quantlib_p else (math.exp(r * dt) - d) / (u - d)
    pay = call_payoff if call else put_payoff
    prices = s * d ** np.arange(steps, -1, -1) * u ** np.arange(steps + 1)
    values = pay(prices, k)
    for _ in range(steps):
        prices = prices[1:] / u
        values = math.exp(-r * dt) * (p * values[1:] + (1 - p) * values[:-1])
        if american:
            values = np.maximum(values, pay(prices, k))
    return float(values[0])

In [ ]:
out["trade"] = {"spot": S, "strike": K, "years": T, "rate_pct": r * 100, "vol_pct": vol * 100}

Part 2's additions: Greeks, simulation, hedging, implied volatility, Black and Bachelier (`episodes/_shared/options_part2.py`):

In [ ]:
# Part 2 additions to options.py: the Greeks, simulation, a hedging simulator, implied volatility, and Black's and
# Bachelier's formulas on a forward (for FX and interest-rate options). Written out by hand and copied verbatim into
# each Part 2 notebook; the notebooks check them against QuantLib. Needs options.py (norm_cdf, black_scholes),
# math and numpy as np.

def norm_pdf(x):
    """The bell curve's height at x (standard normal density)."""
    return math.exp(-x * x / 2) / math.sqrt(2 * math.pi)

def norm_cdf_v(x):
    """norm_cdf for a whole numpy array at once."""
    return 0.5 * (1 + np.vectorize(math.erf)(np.asarray(x, dtype=float) / math.sqrt(2)))

def bs_greeks(s, k, t, r, vol):
    """Price and Greeks of a European call on a share with no dividends.
    delta per $1 of share price, gamma per $1 squared, vega per 1.00 of volatility (100 vol points),
    theta per year (negative: value lost as time passes), rho per 1.00 of interest rate."""
    d1 = (math.log(s / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    d2 = d1 - vol * math.sqrt(t)
    disc = math.exp(-r * t)
    return {"price": s * norm_cdf(d1) - k * disc * norm_cdf(d2),
            "delta": norm_cdf(d1),
            "gamma": norm_pdf(d1) / (s * vol * math.sqrt(t)),
            "vega": s * norm_pdf(d1) * math.sqrt(t),
            "theta": -s * norm_pdf(d1) * vol / (2 * math.sqrt(t)) - r * k * disc * norm_cdf(d2),
            "rho": k * t * disc * norm_cdf(d2)}

def call_delta(s, k, t, r, vol):
    """Black-Scholes delta N(d1) for an array of share prices."""
    d1 = (np.log(np.asarray(s, dtype=float) / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    return norm_cdf_v(d1)

def simulate_paths(s, t, drift, vol, steps, n_paths, rng):
    """Share price paths: shape (n_paths, steps + 1), starting at s, lognormal steps with the given drift."""
    dt = t / steps
    z = rng.standard_normal((n_paths, steps))
    log_steps = (drift - vol ** 2 / 2) * dt + vol * math.sqrt(dt) * z
    return s * np.exp(np.concatenate([np.zeros((n_paths, 1)), np.cumsum(log_steps, axis=1)], axis=1))

def monte_carlo_call(s, k, t, r, vol, n, rng):
    """Price = the average discounted payoff over n risk-neutral outcomes, with its standard error."""
    z = rng.standard_normal(n)
    s_t = s * np.exp((r - vol ** 2 / 2) * t + vol * math.sqrt(t) * z)
    pv = math.exp(-r * t) * np.maximum(s_t - k, 0.0)
    return float(pv.mean()), float(pv.std(ddof=1) / math.sqrt(n))

def hedge_short_call(s, k, t, r, vol_implied, vol_realised, steps, n_paths, rng, hedge=True):
    """Sell the call at vol_implied and, if hedge, rebalance to delta shares (computed at vol_implied) `steps` times.
    The share moves with vol_realised and grows at r. Cash earns r. Returns each path's P&L at expiry in today's dollars."""
    dt = t / steps
    price = s * np.ones(n_paths)
    held = call_delta(price, k, t, r, vol_implied) if hedge else np.zeros(n_paths)
    cash = black_scholes(s, k, t, r, vol_implied) - held * price      # premium received, shares bought
    for i in range(1, steps + 1):
        price = price * np.exp((r - vol_realised ** 2 / 2) * dt + vol_realised * math.sqrt(dt) * rng.standard_normal(n_paths))
        cash = cash * math.exp(r * dt)
        if hedge and i < steps:
            new = call_delta(price, k, t - i * dt, r, vol_implied)
            cash = cash - (new - held) * price                        # buy (or sell) the difference
            held = new
    pnl = cash + held * price - np.maximum(price - k, 0.0)           # sell the shares, pay the option's payoff
    return pnl * math.exp(-r * t)

def implied_vol(price, s, k, t, r, lo=0.0001, hi=3.0, tol=1e-12):
    """Run Black-Scholes backwards by bisection: halve [lo, hi] until the price fits. Returns (vol, halvings)."""
    n = 0
    while hi - lo > tol:
        mid = (lo + hi) / 2
        if black_scholes(s, k, t, r, mid) < price:
            lo = mid
        else:
            hi = mid
        n += 1
    return (lo + hi) / 2, n

def black76(forward, strike, t, vol, df, call=True):
    """Black's formula on a forward: df x [F N(d1) - K N(d2)] for a call. FX options (Garman-Kohlhagen) and caplets."""
    sd = vol * math.sqrt(t)
    d1 = (math.log(forward / strike) + sd ** 2 / 2) / sd
    d2 = d1 - sd
    if call:
        return df * (forward * norm_cdf(d1) - strike * norm_cdf(d2))
    return df * (strike * norm_cdf(-d2) - forward * norm_cdf(-d1))

def bachelier(forward, strike, t, normal_vol, df, call=True):
    """Bachelier's formula: the forward moves by normal_vol x sqrt(t) in absolute terms (it can go below zero)."""
    sd = normal_vol * math.sqrt(t)
    d = (forward - strike) / sd
    if call:
        return df * ((forward - strike) * norm_cdf(d) + sd * norm_pdf(d))
    return df * ((strike - forward) * norm_cdf(-d) + sd * norm_pdf(d))

## 2. From coin flips to paths

Part 1's tree moved the share up or down once per step. Take a walk that starts at zero and, at each of $n$ steps in a year, moves up or down by $\sqrt{1/n}$ with equal chances. Make the steps smaller and more numerous and the walk turns into a jagged, continuous path: **Brownian motion**, written $W_t$.

Why a step of $\sqrt{1/n}$? Each step adds $1/n$ to the variance, so after a year the variance is exactly 1, however many steps we take.

In [ ]:
rng = np.random.default_rng(11)

def coin_walk(steps, n_paths, rng):
    '''n_paths random walks over one year: each step is up or down by sqrt(1/steps), with equal chances.'''
    moves = rng.choice([-1.0, 1.0], size=(n_paths, steps)) * math.sqrt(1 / steps)
    return np.concatenate([np.zeros((n_paths, 1)), moves.cumsum(axis=1)], axis=1)

stages = {}
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2), sharey=True)
for ax, steps in zip(axes, (4, 16, 252)):
    w = coin_walk(steps, 30, rng)
    t = np.linspace(0, 1, steps + 1)
    stages[str(steps)] = {"t": t.round(5).tolist(), "paths": w.round(4).tolist()}
    ax.plot(t, w.T, lw=0.8)
    ax.set_title(f"{steps} steps a year")
out["coins"] = {"stages": stages, "steps": [4, 16, 252], "n_paths": 30,
                "ticks": [{"x": x, "label": lab} for x, lab in ((0, "0"), (0.25, "3m"), (0.5, "6m"), (0.75, "9m"), (1, "1y"))]}
plt.show()

## 3. The spread grows with the square root of time

20,000 simulated paths with daily steps (252 a year, each a normal draw with variance $1/252$). The average stays near zero, but the variance grows in step with time, so the spread (standard deviation) grows with $\sqrt{t}$.

In [ ]:
n_paths, steps = 20_000, 252
dW = rng.standard_normal((n_paths, steps)) * math.sqrt(1 / steps)
W = np.concatenate([np.zeros((n_paths, 1)), dW.cumsum(axis=1)], axis=1)
t = np.arange(steps + 1) / steps
rows = []
for label, day in (("3 months", 63), ("6 months", 126), ("1 year", 252)):
    rows.append({"time": label, "t": day / steps, "mean": float(W[:, day].mean()), "variance": float(W[:, day].var()),
                 "sd": float(W[:, day].std()), "sqrt_t": math.sqrt(day / steps)})
    assert abs(W[:, day].var() / (day / steps) - 1) < 0.03       # variance within 3% of t
bins = np.arange(-3.5, 3.51, 0.25)
counts, _ = np.histogram(W[:, -1], bins=bins)
every = 3   # plot every third day: enough for a smooth line
out["spread"] = {"rows": rows, "n_paths": n_paths, "var_q": rows[0]["variance"], "var_h": rows[1]["variance"], "var_1": rows[2]["variance"],
                 "sd_1": rows[2]["sd"], "mean_1": rows[2]["mean"],
                 "t": t[::every].round(4).tolist(), "paths": W[:40, ::every].round(3).tolist(),
                 "upper": np.sqrt(t[::every]).round(4).tolist(), "lower": (-np.sqrt(t[::every])).round(4).tolist(),
                 "hist": {"y": ((bins[:-1] + bins[1:]) / 2).round(3).tolist(), "share": (counts / n_paths).round(5).tolist()}}
plt.figure(figsize=(8, 3.5))
plt.plot(t, W[:40].T, lw=0.6, alpha=0.6)
plt.plot(t, np.sqrt(t), "k--", t, -np.sqrt(t), "k--")
plt.title("40 of 20,000 paths, with ± one standard deviation (√t)")
plt.show()
pd.DataFrame(rows)

## 4. Infinitely wiggly

Add up the *size* of every move (ignoring its direction): the distance the path travels. With $n$ steps each move is about $\sqrt{1/n}$ in size, so the distance grows like $n \times \sqrt{1/n} = \sqrt{n}$ (exactly $\sqrt{2n/\pi}$ on average). Finer steps, more distance, without limit, while the endpoint stays in the same place (typically within about ±1).

In [ ]:
wiggle = []
for steps in (12, 52, 252, 10_000):
    moves = rng.standard_normal((1_000, steps)) * math.sqrt(1 / steps)
    wiggle.append({"steps": steps, "distance": float(np.abs(moves).sum(axis=1).mean()),
                   "theory": math.sqrt(2 * steps / math.pi), "end_abs": float(np.abs(moves.sum(axis=1)).mean())})
    assert abs(wiggle[-1]["distance"] / wiggle[-1]["theory"] - 1) < 0.02
out["wiggle"] = {"rows": wiggle, "daily": wiggle[2]["distance"], "fine": wiggle[3]["distance"],
                 "monthly": wiggle[0]["distance"], "end_abs": wiggle[2]["end_abs"]}
pd.DataFrame(wiggle)

## 5. Squared moves add up to time

Now add up the *squares* of the moves instead. Take one path with 10,000 steps in a year, and measure it with coarser and coarser steps (every 1,000th point, every 100th, ...). As the steps shrink, the sum of squared moves settles on exactly 1, the time elapsed. This is the **quadratic variation** of Brownian motion, $\sum (\Delta W)^2 \to t$, often written $(dW)^2 = dt$. It is the fact Itô's lemma (Episode 13) rests on.

The last column shows how much the sum wobbles from path to path (its standard deviation over 1,000 paths): $\sqrt{2/n}$, shrinking to zero.

In [ ]:
fine = rng.standard_normal(10_000) * math.sqrt(1 / 10_000)
path = np.concatenate([[0.0], fine.cumsum()])
many = rng.standard_normal((1_000, 10_000)) * math.sqrt(1 / 10_000)
many_paths = np.concatenate([np.zeros((1_000, 1)), many.cumsum(axis=1)], axis=1)
qv = []
for steps in (10, 100, 1_000, 10_000):
    stride = 10_000 // steps
    sq = float((np.diff(path[::stride]) ** 2).sum())
    wobble = float((np.diff(many_paths[:, ::stride], axis=1) ** 2).sum(axis=1).std())
    qv.append({"steps": steps, "this_path": sq, "wobble": wobble, "wobble_theory": math.sqrt(2 / steps)})
assert abs(qv[-1]["this_path"] - 1) < 0.05
out["qv"] = {"rows": qv, "q10": qv[0]["this_path"], "q100": qv[1]["this_path"], "q1000": qv[2]["this_path"],
             "q10000": qv[3]["this_path"]}
pd.DataFrame(qv)

## 6. History

Louis Bachelier modelled prices on the Paris Bourse with random walks in his thesis *Théorie de la spéculation* (Annales scientifiques de l'École Normale Supérieure, 1900), five years before Albert Einstein's 1905 paper on the motion of small particles suspended in a liquid (*Annalen der Physik*), which explained Brownian motion.

In [ ]:
out["history"] = {"bachelier": 1900, "einstein": 1905, "gap": 1905 - 1900}
out["history"]

## 7. This episode's question

If the variance after one year is 1, what is the standard deviation after four years? Variance grows with time, so it is 4 after four years, and the standard deviation is $\sqrt{4} = 2$.

In [ ]:
out["question"] = {"years": 4, "variance": 4 * 1.0, "sd": math.sqrt(4 * 1.0)}
out["question"]

## 8. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")